# 基于大数据的房产估价

### 一、实验目的与要求

    1、掌握使用numpy和pandas库处理数据的基本方法。
    2、掌握使用Sklearn库对多元线性回归算法的实现及其评价方法。
    3、掌握使用matplotlib结合pandas库对数据分析可视化处理的基本方法。

### 二、实验内容

    1、利用python中pandas等库完成对数据的预处理。
    2、利用pandas、matplotlib等库完成对预处理数据的可视化。
    3、结合pandas、matplotlib库对聚类完成的结果进行可视化处理。

### 三、实验步骤

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import os
import matplotlib.pyplot as mpl
# 设置绘图显示中文字体
mpl.rcParams["font.sans-serif"]="SimHei" #设置字体样式
mpl.rcParams["axes.unicode_minus"]=False #设置为字符显示


In [ ]:
input_dir = '/data/bigfiles'
files = os.listdir(input_dir)
files

In [ ]:
data = []
for i in files:
    data.append(pd.read_excel(input_dir +'/'+ i))
data = pd.concat(data)

data.head(1)

In [ ]:
data = data.reset_index()
data = data.drop('index', axis = 1)
print(len(data))
data.head(10)

In [ ]:
 data.房屋信息.map(lambda x:x.split('|'))

In [ ]:
data.房屋信息.map(lambda x:len(x.split('|'))).value_counts()

In [ ]:
data = data[data.房屋信息.map(lambda x:len(x.split('|')) == 7)]
data

In [ ]:
data.shape

In [ ]:
data.loc[:,'户型'] = data.房屋信息.map(lambda x : x.split('|')[0]) 
data.loc[:,'面积'] = data.房屋信息.map(lambda x : x.split('|')[1])
data.loc[:,'朝向'] = data.房屋信息.map(lambda x : x.split('|')[2])
data.loc[:,'类型'] = data.房屋信息.map(lambda x : x.split('|')[3])
data.loc[:,'楼层'] = data.房屋信息.map(lambda x : x.split('|')[4])
data.loc[:,'建成时间'] = data.房屋信息.map(lambda x : x.split('|')[5]) 
data.loc[:,'结构'] = data.房屋信息.map(lambda x : x.split('|')[6]) # 删除房屋信息列
data = data.drop('房屋信息', axis = 1) 

In [ ]:
data.head(1)

In [ ]:
data.关注信息.map(lambda x : len(x.split('/'))).value_counts()

In [ ]:
data['关注人数'] = data.关注信息.map(lambda x : x.split('/')[0])
data['发布时间'] = data.关注信息.map(lambda x : x.split('/')[1])

In [ ]:
data.楼层.map(lambda x : len(x.split('('))).value_counts()

In [ ]:
data = data[data.楼层.map(lambda x : len(x.split('(')) == 2)]

In [ ]:
data.head(1)

In [ ]:
data['所在楼层'] = data.楼层.map(lambda x : x.split('(')[0])
data['总楼层'] = data.楼层.map(lambda x : x.split('(')[1])

In [ ]:
data.head(1)

In [ ]:
data = data.drop(['关注信息','楼层'],axis = 1)

In [ ]:
data.head(1)

In [ ]:
data.shape

In [ ]:
data = data.reset_index()
data = data.drop('index', axis = 1)
data.head()

In [ ]:
data.describe()

In [ ]:
data.单价 = data.单价.map(lambda x: int(re.findall('\d+',x)[0])/10000.0).round(2)
def extract_number(series):
    return (
        series.astype(str)  # 确保所有值为字符串
        .str.extract('(\d+)', expand=False)  # 提取数字部分
        .astype(float)  # 转为浮点数
    )

# 应用处理
data.面积 = extract_number(data.面积)
data.建成时间 = extract_number(data.建成时间).astype('Int64')  # 转为可空整数
data.关注人数 = extract_number(data.关注人数).astype('Int64')
data.总楼层 = extract_number(data.总楼层).astype('Int64')
data = data.reindex()
data.head()

In [ ]:
data = data.join(pd.get_dummies(data.区域))
data.户型.unique()

In [ ]:
data = data.join(pd.get_dummies(data.户型))
data.结构.unique()

In [ ]:
data = data.drop(data[data.结构 == ' 暂无数据'].index)
data.结构 = data.结构.map(lambda x: x.strip())
data.结构.unique()

In [ ]:
data = data.join(pd.get_dummies(data.结构))
data.head()

In [ ]:
data.所在楼层.unique()

In [ ]:
data = data.drop(data[data.所在楼层 == ' 未知'].index)
data.所在楼层 = data.所在楼层.map(lambda x: x.strip())
data = data.join(pd.get_dummies(data.所在楼层))

In [ ]:
data.类型.unique()

In [ ]:
data.类型 = data.类型.map(lambda x: x.strip())
data = data.join(pd.get_dummies(data.类型))
data.shape

In [ ]:
data.columns

In [ ]:
data = data.drop(['户型','类型','结构','发布时间','所在楼层'], axis = 1)

In [ ]:
data.head()

In [ ]:
data.shape

In [ ]:
data = data.reset_index()
data = data.drop('index',axis = 1)

In [ ]:
data.index

In [ ]:
data.朝向.map(lambda x: len(x.split())).value_counts()

In [ ]:
array1 = np.zeros((data.shape[0],8))
array1

In [ ]:
df1 = pd.DataFrame(array1,columns = ['东','南','西','北','东北','东南','西南','西北'])
df1

In [ ]:
for i in range(len(data.朝向)):
    for j in data.朝向[i].strip().split():
        df1.loc[i,j] = 1

In [ ]:
df1.sum(axis = 1).value_counts()

In [ ]:
data = data.join(df1)
data = data.drop('朝向', axis = 1)
data.shape

In [ ]:
data.head()

In [ ]:
data.describe()

In [ ]:
#各区域平均房价
(data.pivot_table(values='单价', 
                 index='区域', 
                 aggfunc='mean')
 .sort_values(by='单价', ascending=False)
 .plot(kind='barh', 
       color="c",
       figsize=(10, 6),
       title='各区域平均单价对比'))

# 添加标签和网格线
plt.xlabel('单价(万元)')  # 注意这里没有缩进
plt.ylabel('区域')
plt.grid(axis='x', linestyle='--')

# 显示图表
plt.tight_layout()
plt.show()

In [ ]:
#单价top10
ntop = 10
data_top = data.sort_values(by='单价', ascending=False)[:ntop]
y = range(len(data_top.单价))
width = data_top.单价
y_label = data_top.位置信息 + '(' + data_top.区域 + ')'

fig = plt.figure(figsize=(12, 4))
ax = fig.add_subplot(121)

ax.set(title='单价top10') 

ax.barh(y, width, 
        facecolor='bisque',
        edgecolor='pink',
        height=0.5,
        tick_label=y_label) 

plt.tight_layout()
plt.show()

In [ ]:
#房屋朝向分布
data[['东','南','西','北','东南','西南','东北','西北']].sum().plot(kind='bar')
plt.ylabel(' 数量')
plt.xticks(rotation=0) 

In [ ]:
#绘制某一区域的价格所在
bins = [0,0.3,0.6,0.9,1.2,1.5,1.8,2.1,2.5]
plt.xlabel("价格区间")
pd.cut(data[data['区域' ]=='双流'].单价, bins).value_counts().plot(kind='bar')

In [ ]:
#单价与关注人数散点图
data.plot(kind="scatter",x="单价",y="关注人数",alpha=0.4)

In [ ]:
#单价与面积散点图
data.plot(kind="scatter",x="单价",y="面积",alpha=0.4)

In [ ]:
#建立回归模型
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression as LR

In [ ]:
data.columns

In [ ]:
total_price = data.总价
unit_price = data.单价
house_area = data.面积
house_type = data[[ '0室0厅 ', '0室1厅 ',
       '1室0厅 ', '1室1厅 ', '1室2厅 ', '2室0厅 ', '2室1厅 ', '2室2厅 ', '3室0厅 ', '3室1厅 ',
       '3室2厅 ', '3室3厅 ', '3室4厅 ', '4室0厅 ', '4室1厅 ', '4室2厅 ', '4室3厅 ', '4室4厅 ',
       '5室0厅 ', '5室1厅 ', '5室2厅 ', '5室3厅 ', '5室4厅 ', '6室1厅 ', '6室2厅 ', '6室3厅 ',
       '6室4厅 ', '7室1厅 ', '7室2厅 ', '7室3厅 ', '7室4厅 ', '7室5厅 ', '8室2厅 ', '8室3厅 ',
       '9室2厅 ']]
house_class = data[['毛坯', '简装', '精装']]
house_dirt = data[['东', '南', '西', '北', '东北',
       '东南', '西南', '西北']]
house_layer = data[[ '中楼层', '低楼层',
       '地下室',  '高楼层']]
total_layer = data.总楼层
X = pd.concat([house_area, house_type, house_class, house_dirt, house_layer, total_layer], axis=1)
Y = unit_price

In [ ]:
Xtrain,Xtest,Ytrain,Ytest = train_test_split(X,Y,test_size=0.2, random_state = 10)

In [ ]:
reg = LR().fit(Xtrain, Ytrain)
yhat = reg.predict(Xtest)
print(list(zip(X.columns, reg.coef_)))

In [ ]:
n = 50
plt.plot(range(len(yhat[:n])),yhat[:n])
plt.plot(range(len(Ytest[:n])),Ytest[:n])
plt.xlabel('个例')
plt.ylabel('单价')
plt.title('线性回归预测结果')
plt.legend(["预估","实际"])

In [ ]:
# 用于检验模型效果
from sklearn.metrics import mean_squared_error  # MSE
from sklearn.metrics import mean_absolute_error  # MAE
from sklearn.metrics import r2_score  # R2

mse = mean_squared_error(Ytest, yhat)  # MSE
mae = mean_absolute_error(Ytest, yhat)  # MAE
r2 = r2_score(Ytest, yhat)  # R2
# 调整R2
n = Xtest.shape[0]
k = Xtest.shape[1]
adj_r2 = 1-(1-r2)*((n-1)/(n-k-1))

print('MSE : ' + str(mse))
print('MAE : ' + str(mae))
print('R2 : ' + str(r2))
print('调整R2 ：' + str(adj_r2))


### 四、个人总结